# Multi-aspect model (ViDeBERTa + Adapter + Multi-Pool + MSD)

<!-- Notebook này: EDA nhẹ trên nhãn và **smoke test** một bước forward. Huấn luyện đầy đủ nên chạy `python -m model.train` từ thư mục `src` (xem `ML_README.md`). -->

In [1]:
from pathlib import Path
import os
import sys


def resolve_repo_root() -> Path:
    env_root = os.getenv("ML_REPO_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "src" / "model").is_dir():
            return p
        raise FileNotFoundError(f"ML_REPO_ROOT is invalid: {p}")

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "src" / "model").is_dir():
            return p
    raise FileNotFoundError("Cannot locate repo root. Set ML_REPO_ROOT if needed.")


REPO = resolve_repo_root()
src_root = REPO / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from model.config import TrainConfig
from model.data import AspectReviewDataset, collate_batch, load_review_frame
from model.videberta_msd import ViDeBERTaAspectMSD

d:\Education\Semester 8\Machine Learning\Lab\Lab 01\ML-23KHDL1-Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and simple eda

In [2]:
DATA = REPO / "data" / "cleaned" / "crawl_only_train.csv"
if not DATA.is_file():
    raise FileNotFoundError(f"Missing file: {DATA}")

# Load data and print quick label distribution
df = pd.read_csv(DATA)
label_cols = [
    "vệ sinh_label",
    "đồ ăn thức uống_label",
    "khách sạn_label",
    "vị trí_label",
    "phòng ốc_label",
    "dịch vụ_label",
]
print("rows", len(df))
for c in label_cols:
    print(c, df[c].value_counts().sort_index().to_dict())

rows 13224
vệ sinh_label {0: 9309, 1: 3155, 2: 760}
đồ ăn thức uống_label {0: 10960, 1: 1731, 2: 533}
khách sạn_label {0: 3108, 1: 9074, 2: 1042}
vị trí_label {0: 8436, 1: 4507, 2: 281}
phòng ốc_label {0: 7655, 1: 3914, 2: 1655}
dịch vụ_label {0: 6897, 1: 5782, 2: 545}


In [3]:
# # Smoke test only: one forward pass to verify data/tokenizer/model pipeline
# SMOKE_ROWS = 8
# SMOKE_BATCH_SIZE = 2
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# cfg = TrainConfig()
# tok = AutoTokenizer.from_pretrained(cfg.model_name)
# train_csv = REPO / "data" / "cleaned" / "crawl_only_train.csv"
# if not train_csv.is_file():
#     raise FileNotFoundError(f"Missing file: {train_csv}")
# smoke_df = load_review_frame(train_csv).head(SMOKE_ROWS)

# ds = AspectReviewDataset(smoke_df, tok, cfg.max_length)
# loader = DataLoader(
#     ds,
#     batch_size=SMOKE_BATCH_SIZE,
#     shuffle=False,
#     collate_fn=collate_batch,
#     num_workers=0,
#     )
# batch = next(iter(loader))
# batch = {k: v.to(device) for k, v in batch.items()}

# model = ViDeBERTaAspectMSD(cfg).to(device)
# model.train()
# out_train = model(
#     input_ids=batch["input_ids"],
#     attention_mask=batch["attention_mask"],
#     labels=batch["labels"],
#     )
# print(f"[SMOKE] device={device}")
# print(f"[SMOKE] train loss: {float(out_train['loss'].detach()):.6f}")
# print(f"[SMOKE] train logits shape: {tuple(out_train['logits'].shape)}")

# model.eval()
# with torch.no_grad():
#     out_eval = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
# print(f"[SMOKE] eval logits shape: {tuple(out_eval['logits'].shape)}")
# print("[SMOKE] This cell does NOT train full model. For full training run: python -m model.train")

[SMOKE] device=cpu
[SMOKE] train loss: 1.091303
[SMOKE] train logits shape: (2, 6, 3)
[SMOKE] eval logits shape: (2, 6, 3)
[SMOKE] This cell does NOT train full model. For full training run: python -m model.train


In [ ]:
# Full training runner (actual training, not smoke test)
import os
import subprocess
import sys
from pathlib import Path

SRC_DIR = REPO / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Missing src dir: {SRC_DIR}")

# Tune these before running
EPOCHS = 5
BATCH_SIZE = 16
NUM_WORKERS = 2
LR = 3e-4
SEED = 42
EXTRA_ARGS = []  # Example: ["--no-multipool"] or ["--msd-single-p", "0.1"]

cmd = [
    sys.executable,
    "-m",
    "model.train",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--seed", str(SEED),
    *EXTRA_ARGS,
]

print("Running:", " ".join(cmd))
print("Working directory:", SRC_DIR)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    cmd,
    cwd=str(SRC_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Training failed with exit code {ret}")

best_ckpt = REPO / "models" / "best.pt"
if best_ckpt.is_file():
    print(f"\nDone. Best checkpoint: {best_ckpt}")
else:
    print("\nDone. Check logs above for save path.")

<!-- ## Ablation cho báo cáo

Chạy từ thư mục `src` (xem `ML_README.md`):

- **Chỉ CLS, không multi-pool / MSD**: `python -m model.train --no-multipool --no-msd`
- **Multi-pool, MSD một tỷ lệ (0.1 × 5)**: `python -m model.train --msd-single-p 0.1`
- **Đầy đủ (multi-pool + multi-rate)**: `python -m model.train`

So sánh **macro-F1 trung bình 6 khía cạnh** trên validation (in log sau mỗi epoch). -->